In [ ]:
import data_loader
import numpy as np
from skimage.feature import hog
from skimage import color
from sklearn.preprocessing import StandardScaler
from sklearn import svm
from sklearn.metrics import f1_score, roc_curve, auc, precision_recall_curve, confusion_matrix, classification_report
from multiprocessing import Pool, cpu_count
import time
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm


In [ ]:
train, val, test = data_loader.load_masks()


In [ ]:
def sliding_window(image, step_size, window_size):
    for y in range(0, image.shape[0], step_size):
        for x in range(0, image.shape[1], step_size):
            yield (x, y, image[y:y+window_size[1], x:x+window_size[0]])


In [ ]:
def extract_hog_features(window):
    """Extract HOG features from a window patch."""
    if window.size == 0:
        return np.zeros(324)  # Default HOG feature size for 64x64 window
    
    # Convert to grayscale
    if len(window.shape) == 3:
        gray = color.rgb2gray(window)
    else:
        gray = window
    
    # Extract HOG features
    features = hog(gray, orientations=9, pixels_per_cell=(8, 8), 
                   cells_per_block=(2, 2), feature_vector=True)
    
    return features

def process_window_hog(args):
    """Process a single window for parallel execution."""
    x, y, window_patch, mask_patch = args
    feature_vector = extract_hog_features(window_patch)
    label = 1 if np.sum(mask_patch > 0) > (mask_patch.size * 0.1) else 0
    return feature_vector, label


In [ ]:
# Parallel HOG feature extraction
print(f"Using {cpu_count()} CPU cores for parallel processing...")
start_time = time.time()

# Collect all windows and mask patches first
all_windows_data = []
print("Collecting windows from training images...")
for data in tqdm(train, desc="Collecting windows"):
    image = data['image']
    mask = data['mask']
    
    windows = list(sliding_window(image, 32, (64, 64)))
    for x, y, window_patch in windows:
        h, w = window_patch.shape[:2]
        mask_patch = mask[y:y+h, x:x+w]
        all_windows_data.append((x, y, window_patch, mask_patch))

print(f"Total windows to process: {len(all_windows_data)}")

# Process in parallel with progress bar
n_jobs = min(cpu_count(), 8)  # Limit to 8 to avoid overhead
results = []

with Pool(n_jobs) as pool:
    # Use imap_unordered for better progress updates
    with tqdm(total=len(all_windows_data), desc="Extracting HOG features") as pbar:
        for result in pool.imap_unordered(process_window_hog, all_windows_data):
            results.append(result)
            pbar.update(1)

# Unpack results
all_features, all_labels = zip(*results)
X = np.array(all_features)
y = np.array(all_labels)

elapsed = time.time() - start_time
print(f"Extracted {len(X)} feature vectors in {elapsed:.1f} seconds ({elapsed/60:.1f} minutes)")
print(f"Feature vector dimension: {X.shape[1]}")
print(f"Class distribution: {np.bincount(y)}")


In [ ]:
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
# Train SVM classifier
clf = svm.SVC()
clf.fit(X_scaled, y)
print("Model trained!")


In [ ]:
def predict_image(image, model, scaler, step_size=32, window_size=(64, 64), n_jobs=None):
    """Predict on a single image using sliding window with HOG features (parallelized)."""
    if n_jobs is None:
        n_jobs = min(cpu_count(), 8)
    
    windows = list(sliding_window(image, step_size, window_size))
    
    # Extract HOG features in parallel
    window_patches = [w[2] for w in windows]
    with Pool(n_jobs) as pool:
        features = pool.map(extract_hog_features, window_patches)
    
    features = np.array(features)
    
    # Scale and predict
    features_scaled = scaler.transform(features)
    predictions = model.predict(features_scaled)
    
    # Reconstruct mask
    mask = np.zeros((image.shape[0], image.shape[1]), dtype=np.uint8)
    for (x, y, _), pred in zip(windows, predictions):
        if pred == 1:  # damage
            h, w = window_size[1], window_size[0]
            mask[y:y+h, x:x+w] = 1
    
    return mask


In [ ]:
# Extract test features and evaluate
print("Extracting test features...")
start_time = time.time()

test_windows_data = []
print("Collecting windows from test images...")
for data in tqdm(test, desc="Collecting test windows"):
    image = data['image']
    mask = data['mask']
    
    windows = list(sliding_window(image, 32, (64, 64)))
    for x, y, window_patch in windows:
        h, w = window_patch.shape[:2]
        mask_patch = mask[y:y+h, x:x+w]
        test_windows_data.append((x, y, window_patch, mask_patch))

print(f"Total test windows: {len(test_windows_data)}")

# Process in parallel with progress bar
test_results = []
with Pool(n_jobs) as pool:
    with tqdm(total=len(test_windows_data), desc="Extracting test HOG features") as pbar:
        for result in pool.imap_unordered(process_window_hog, test_windows_data):
            test_results.append(result)
            pbar.update(1)

test_features, test_labels = zip(*test_results)
X_test = np.array(test_features)
y_test = np.array(test_labels)

X_test_scaled = scaler.transform(X_test)

elapsed = time.time() - start_time
print(f"Test features extracted in {elapsed:.1f} seconds")

# Get predictions
predictions = clf.predict(X_test_scaled)
decision_scores = clf.decision_function(X_test_scaled)


In [ ]:
# Comprehensive evaluation metrics
print("=" * 60)
print("EVALUATION METRICS")
print("=" * 60)

# F1 Score
f1 = f1_score(y_test, predictions)
print(f"\nF1 Score: {f1:.4f}")

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, predictions, target_names=['Background', 'Damage']))

# Confusion Matrix
cm = confusion_matrix(y_test, predictions)
print("\nConfusion Matrix:")
print(cm)

# Plot confusion matrix
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Background', 'Damage'],
            yticklabels=['Background', 'Damage'])
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, decision_scores)
roc_auc = auc(fpr, tpr)

plt.subplot(1, 2, 2)
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

print(f"\nROC AUC: {roc_auc:.4f}")

# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, decision_scores)
pr_auc = auc(recall, precision)

plt.figure(figsize=(6, 5))
plt.plot(recall, precision, color='blue', lw=2, label=f'PR curve (AUC = {pr_auc:.4f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc="lower left")
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"PR AUC: {pr_auc:.4f}")
print("=" * 60)
